# nb04b — Finish Experiment A (speed + log) + Run Experiment B
**UCU AI 2026 — HW2**

## Context
Experiment A (`rpn_pre_nms_top_n_train=2000`) finished in the previous notebook:
- mAP: 0.1561  (baseline: 0.1475, Δ = +0.0086)
- AP_small: 0.0963  (baseline: 0.0803, +20% relative)
The previous notebook crashed in `benchmark_speed` due to a CPU-vs-GPU tensor mismatch.
The trained weights `fasterrcnn_tuneA_rpn2000.pth` and `val_preds_A.json` are in the
`suchok/outputs-3` Kaggle dataset.

## Plan
1. Bootstrap with the fixed `benchmark_speed` (GPU-only, no CPU pass)
2. Load A's weights, re-run the speed benchmark, log the final metrics to W&B
3. Train Experiment B (`box_positive_fraction=0.25 → 0.50`, warm-started from baseline)
4. Evaluate B on full val, save `fasterrcnn_tuneB_posfrac05.pth`
5. Build comparison table, per-class AP, duplicate-rate analysis
6. Generate and save the Kaggle submission with the winner

## 1. Bootstrap — same environment as nb04, plus a GPU-only `benchmark_speed`

`benchmark_speed` is overridden here so it does **not** try to run the model on CPU
(that was the crash). Only GPU ms/image and FPS are produced; those are the only
numbers the report needs.

In [1]:
# =============================================================================
# nb04b bootstrap — reconstructs nb04's environment
# NOTE: this dataset now also contains outputs-3 (Exp A weights + preds)
# =============================================================================
import sys, json, time, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm.auto import tqdm

from torchvision.ops import box_iou
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection._utils import Matcher

# ---- Paths -----------------------------------------------------------------
COMP_DIR     = Path("/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026")
DATASET_DIR  = Path("/kaggle/input/datasets/suchok/my-dataset")
OUTPUTS_DIR  = Path("/kaggle/input/datasets/suchok/outputs")            # nb02 artifacts
OUTPUTS3_DIR = Path("/kaggle/input/datasets/suchok/outputs-3")          # nb04 Exp A artifacts
WEIGHTS_DIR  = Path("/kaggle/input/datasets/suchok/baseline-weights")   # nb03 baseline
WORK_DIR     = Path("/kaggle/working")

assert COMP_DIR.exists(),     f"missing competition: {COMP_DIR}"
assert DATASET_DIR.exists(),  f"missing my-dataset: {DATASET_DIR}"
assert OUTPUTS_DIR.exists(),  f"missing outputs: {OUTPUTS_DIR}"
assert OUTPUTS3_DIR.exists(), f"missing outputs-3: {OUTPUTS3_DIR}"
assert WEIGHTS_DIR.exists(),  f"missing baseline-weights: {WEIGHTS_DIR}"

WORK_DIR.mkdir(parents=True, exist_ok=True)

# ---- common.py --------------------------------------------------------------
sys.path.insert(0, str(DATASET_DIR))
import common, importlib
importlib.reload(common)
from common import (
    set_seeds, SEED, CLASS_NAMES, NUM_CLASSES, IMAGENET_MEAN, IMAGENET_STD,
    load_splits, index_split, filter_by_clips,
    load_yolo_labels, load_ignore_regions,
    build_fullval_gt, evaluate_full_val,
    predict_with_strategy, save_metrics, save_checkpoint,
    detection_collate, VisDroneDetectionDataset,
    predict_test_and_write_submission, read_image_sizes,
    coco_eval_summary,
)

set_seeds(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE, "| torch:", torch.__version__)

# ---- Locked protocol --------------------------------------------------------
chosen = json.load(open(OUTPUTS_DIR / "chosen_preprocessing.json"))
STRAT  = chosen["strategy"]
SIZE   = int(chosen["size"])
print("Locked preprocessing:", STRAT, SIZE)

train_clips, val_clips = load_splits(OUTPUTS_DIR / "splits")
assert set(train_clips).isdisjoint(set(val_clips))
print(f"clips: train={len(train_clips)}  val={len(val_clips)}")

TRAIN_DIR = COMP_DIR / "train"
VAL_DIR   = COMP_DIR / "val"
TEST_CSV  = COMP_DIR / "test_images.csv"

train_index = filter_by_clips(index_split(TRAIN_DIR), train_clips)
val_index   = filter_by_clips(index_split(VAL_DIR),   val_clips)
print(f"frames: train={len(train_index):,}  val={len(val_index):,}")

# ---- Full-val GT ------------------------------------------------------------
gt_fullval_ignore   = WORK_DIR / "gt_fullval_ignore.json"
gt_fullval_noignore = WORK_DIR / "gt_fullval_noignore.json"

for src_dir in (WEIGHTS_DIR, OUTPUTS_DIR):
    for name in ("gt_fullval_ignore.json", "gt_fullval_noignore.json"):
        if (src_dir / name).exists() and not (WORK_DIR / name).exists():
            shutil.copy(src_dir / name, WORK_DIR / name)
            print(f"copied {name} from {src_dir}")

if not gt_fullval_ignore.exists():
    build_fullval_gt(val_index, str(gt_fullval_ignore), include_ignore=True)
if not gt_fullval_noignore.exists():
    build_fullval_gt(val_index, str(gt_fullval_noignore), include_ignore=False)

# ---- Model builder ----------------------------------------------------------
def build_fasterrcnn(rpn_pre_nms_train=1000, rpn_post_nms_train=1000,
                     rpn_pre_nms_test=1000,  rpn_post_nms_test=300,
                     box_positive_fraction=0.25, box_batch_size=512,
                     score_thresh=0.05, nms_thresh=0.5):
    m = fasterrcnn_resnet50_fpn_v2(
        weights=None, weights_backbone=None,
        min_size=SIZE, max_size=SIZE,
        rpn_pre_nms_top_n_train=rpn_pre_nms_train,
        rpn_post_nms_top_n_train=rpn_post_nms_train,
        rpn_pre_nms_top_n_test=rpn_pre_nms_test,
        rpn_post_nms_top_n_test=rpn_post_nms_test,
        box_batch_size_per_image=box_batch_size,
        box_positive_fraction=box_positive_fraction,
        box_score_thresh=score_thresh,
        box_nms_thresh=nms_thresh,
    )
    in_feat = m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor = FastRCNNPredictor(in_feat, NUM_CLASSES + 1)
    return m

# ---- Ignore-aware assigner (identical to nb03/nb04) ------------------------
class IgnoreAwareAssigner(torch.nn.Module):
    def __init__(self, high_thresh=0.5, low_thresh=0.4, ignore_iou_thresh=0.3):
        super().__init__()
        self.matcher = Matcher(high_thresh, low_thresh, allow_low_quality_matches=True)
        self.ignore_iou_thresh = ignore_iou_thresh
        self.current_ignore_boxes = None
    def forward(self, anchors, gt_boxes):
        matched_labels, matched_boxes = self.matcher(anchors, gt_boxes)
        if self.current_ignore_boxes is None or len(self.current_ignore_boxes) == 0:
            return matched_labels, matched_boxes
        if len(gt_boxes) == 0:
            return matched_labels, matched_boxes
        matched_labels = matched_labels.clone()
        iou = box_iou(anchors, self.current_ignore_boxes).max(dim=1).values
        ignored = (iou > self.ignore_iou_thresh) & (matched_labels >= 0)
        matched_labels[ignored] = -1
        return matched_labels, matched_boxes

BASELINE_CKPT = WEIGHTS_DIR / "fasterrcnn_baseline_last.pth"
assert BASELINE_CKPT.exists()

def load_checkpoint(m, ckpt, device=DEVICE):
    """Attach ignore-aware assigner, then load a checkpoint with strict=False."""
    m.rpn.box_assigner = IgnoreAwareAssigner().to(device)
    state = torch.load(ckpt, map_location=device)
    missing, unexpected = m.load_state_dict(state, strict=False)
    missing    = [k for k in missing    if "box_assigner" not in k]
    unexpected = [k for k in unexpected if "box_assigner" not in k]
    if missing:    print("WARN missing keys:",    missing[:5], "...")
    if unexpected: print("WARN unexpected keys:", unexpected[:5], "...")
    m.to(device)
    print(f"loaded checkpoint: {Path(ckpt).name}")
    return m

# ---- FIXED benchmark_speed (GPU-only, no CPU pass) -------------------------
def benchmark_speed_gpu(model, index_df, strategy, size, n=50,
                        device=None, score_thr=0.5, warmup=5):
    model.eval()
    if device is None:
        device = next(model.parameters()).device
    model.to(device)
    times = []
    for i in range(warmup + n):
        img_path = index_df.iloc[i % len(index_df)]["image_path"]
        if str(device) == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        _ = predict_with_strategy(model, img_path, strategy, size,
                                  score_thr=score_thr, device=device)
        if str(device) == "cuda":
            torch.cuda.synchronize()
        if i >= warmup:
            times.append(time.time() - t0)
    mean_s = float(np.mean(times)); std_s = float(np.std(times))
    return {
        "device":         str(device),
        "gpu_ms_per_image": mean_s * 1000,
        "gpu_fps":        1.0 / mean_s,
        "std_ms":         std_s * 1000,
        "n":              n,
    }

# ---- W&B --------------------------------------------------------------------
import wandb
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
    print("W&B login OK")
except Exception as e:
    print(f"WARN W&B secret unavailable: {e}")

print("=" * 60)
print("bootstrap OK — run cells 3+")
print("=" * 60)

DEVICE: cuda | torch: 2.11.0+cu130
Locked preprocessing: letterbox 1280
clips: train=56  val=7
frames: train=24,201  val=2,846
copied gt_fullval_ignore.json from /kaggle/input/datasets/suchok/baseline-weights
copied gt_fullval_noignore.json from /kaggle/input/datasets/suchok/baseline-weights


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: suchok-pn (suchok-pn-ukrainian-catholic-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login OK
bootstrap OK — run cells 3+


## 2. Finish Experiment A (speed + W&B log)

The previous run crashed only on CPU speed benchmarking; **training, eval, and weight-saving
all completed**. This cell re-loads A's weights, benchmarks GPU speed on 50 val images, and
logs the final metrics into the same W&B run that was left unfinished.

Recorded metrics (from the previous session, so we don't re-evaluate):
- mAP: 0.1561, AP50: 0.3234, AP75: 0.1335
- AP_small: 0.0963, AP_medium: 0.1656, AP_large: 0.2706

In [2]:
# Recorded from the previous run's Cell 6 output
metrics_a = {
    "mAP":      0.15612895963474377,
    "AP50":     0.3233825797467062,
    "AP75":     0.13351126774659625,
    "AP_small": 0.09629251255084864,
    "AP_medium":0.16556544121326644,
    "AP_large": 0.27058399571028974,
}
print("Experiment A metrics (recovered):", json.dumps(metrics_a, indent=2))

# Copy A's weights into WORK_DIR for convenience
EXP_A_CKPT_SRC = OUTPUTS3_DIR / "fasterrcnn_tuneA_rpn2000.pth"
EXP_A_CKPT     = WORK_DIR / "fasterrcnn_tuneA_rpn2000.pth"
if EXP_A_CKPT_SRC.exists() and not EXP_A_CKPT.exists():
    shutil.copy(EXP_A_CKPT_SRC, EXP_A_CKPT)
    print("copied Exp A weights from outputs-3")

if not EXP_A_CKPT.exists():
    raise FileNotFoundError(
        f"Missing Exp A weights. Expected in {EXP_A_CKPT_SRC} "
        "or already in /kaggle/working/.")

# Copy A's val preds too (needed later for per-class AP / duplicate rate)
EXP_A_PREDS_SRC = OUTPUTS3_DIR / "val_preds_A.json"
EXP_A_PREDS     = WORK_DIR / "val_preds_A.json"
if EXP_A_PREDS_SRC.exists() and not EXP_A_PREDS.exists():
    shutil.copy(EXP_A_PREDS_SRC, EXP_A_PREDS)
    print("copied Exp A predictions from outputs-3")

# Rebuild Exp A model, load weights, benchmark GPU speed
set_seeds(SEED)
model_a = load_checkpoint(build_fasterrcnn(rpn_pre_nms_train=2000), EXP_A_CKPT)
model_a.eval()

speed_a = benchmark_speed_gpu(model_a, val_index, STRAT, SIZE, n=50, device=DEVICE)
print("Variant A speed:", speed_a)

# Resume the same W&B run A so all metrics live in one place
run_a = wandb.init(
    project="ucu-hw2-detection",
    name="frnn_tuneA_rpn2000",
    id="mq3z3goe",         # same run id as the previous session
    resume="allow",
    reinit=True,
)
run_a.log({"val/mAP": metrics_a["mAP"], "val/AP50": metrics_a["AP50"],
           "val/AP75": metrics_a["AP75"],
           "val/AP_small": metrics_a["AP_small"],
           "val/AP_medium": metrics_a["AP_medium"],
           "val/AP_large": metrics_a["AP_large"],
           "speed/gpu_fps": speed_a["gpu_fps"]})
run_a.summary["val/mAP"]       = metrics_a["mAP"]
run_a.summary["val/AP_small"]  = metrics_a["AP_small"]
run_a.summary["speed/gpu_fps"] = speed_a["gpu_fps"]
run_a.finish()
print("Experiment A: W&B run resumed and finished.")

del model_a
torch.cuda.empty_cache()

Experiment A metrics (recovered): {
  "mAP": 0.15612895963474377,
  "AP50": 0.3233825797467062,
  "AP75": 0.13351126774659625,
  "AP_small": 0.09629251255084864,
  "AP_medium": 0.16556544121326644,
  "AP_large": 0.27058399571028974
}
copied Exp A weights from outputs-3
copied Exp A predictions from outputs-3
loaded checkpoint: fasterrcnn_tuneA_rpn2000.pth
Variant A speed: {'device': 'cuda', 'gpu_ms_per_image': 352.01525688171387, 'gpu_fps': 2.8407859615471884, 'std_ms': 5.111523477924631, 'n': 50}


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: setting up run mq3z3goe
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261106_003308-mq3z3goe
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run frnn_tuneA_rpn2000
wandb: ⭐️ View project at https://wandb.ai/suchok-pn-ukrainian-catholic-university/ucu-hw2-detection
wandb: 🚀 View run at https://wandb.ai/suchok-pn-ukrainian-catholic-university/ucu-hw2-detection/runs/mq3z3goe
wandb: updating run metadata
wandb: uploading requirements.txt; uploading wandb-summary.json; uploading wandb-metadata.json
wandb: uploading history steps 124-124, summary
wandb: 
wandb: Run history:
wandb: speed/gpu_fps ▁
wandb:      val/AP50 ▁
wandb:      val/AP75 ▁
wandb:  val/AP_large ▁
wandb: val/AP_medium ▁
wandb:  val/AP_small ▁
wandb:       val/mAP ▁
wandb: 
wandb: Run summary:
wandb:                      spee

Experiment A: W&B run resumed and finished.


## 3. Training loop (identical to nb04)

Same 1-epoch SGD protocol. Loss logging switched to `.item()` to remove the harmless
`requires_grad` warning we saw in Experiment A.

In [3]:
from torch.utils.data import DataLoader
from torch.optim import SGD
from torch.optim.lr_scheduler import StepLR

def train_one_epoch(model, loader, optimizer, scheduler, device,
                    wandb_run=None, log_every=100):
    model.train()
    losses_log = {"loss_classifier": [], "loss_box_reg": [],
                  "loss_objectness": [], "loss_rpn_box_reg": []}
    t0 = time.time()
    for step, (images, targets) in enumerate(loader):
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()

        for k in losses_log:
            losses_log[k].append(loss_dict[k].item())     # no requires_grad warning

        if step % log_every == 0:
            total = sum(v[-1] for v in losses_log.values())
            print(f"step {step:5d} | total {total:.4f} | "
                  + " ".join(f"{k}={v[-1]:.3f}" for k, v in losses_log.items())
                  + f" | {time.time()-t0:.0f}s")
            if wandb_run:
                wandb_run.log({"train/total": total,
                               **{f"train/{k}": v[-1] for k, v in losses_log.items()},
                               "train/step": step})
    return {k: float(np.mean(v)) for k, v in losses_log.items()}

## 4. Experiment B — `box_positive_fraction: 0.25 → 0.50`

**Hypothesis:** with 512 RoIs per image, the baseline uses ~128 positives and ~384 negatives.
Doubling positives gives ~2× gradient signal on rare classes (van 6.3%, truck 5.8%) without
changing anything else.

**Expected:** van/truck per-class AP up; small-object AP roughly unchanged (this knob does
not affect proposal quality).

Warm-started from the **baseline** checkpoint (not from Exp A) so B is a clean single-variable
comparison against the same starting point as A.

In [4]:
set_seeds(SEED)
run_b = wandb.init(project="ucu-hw2-detection",
                   name="frnn_tuneB_posfrac05",
                   config={"rpn_pre_nms_top_n_train": 1000,
                           "rpn_post_nms_top_n_train": 1000,
                           "box_positive_fraction": 0.5,
                           "epochs": 1, "lr": 0.005, "batch": 2,
                           "preproc": f"{STRAT}-{SIZE}"},
                   reinit=True)
print("W&B B:", run_b.url)

model_b = load_checkpoint(build_fasterrcnn(box_positive_fraction=0.5), BASELINE_CKPT)

ds_train_b = VisDroneDetectionDataset(train_index, STRAT, SIZE, is_train=False)
loader_b = DataLoader(ds_train_b, batch_size=2, shuffle=True,
                      num_workers=2, collate_fn=detection_collate)

opt_b = SGD([p for p in model_b.parameters() if p.requires_grad],
            lr=0.005, momentum=0.9)
sch_b = StepLR(opt_b, step_size=3, gamma=0.1)

losses_b = train_one_epoch(model_b, loader_b, opt_b, sch_b, DEVICE, run_b)
print("mean train losses B:", losses_b)

save_checkpoint(model_b, WORK_DIR / "fasterrcnn_tuneB_posfrac05.pth")
run_b.log({"train/mean_losses": losses_b})

wandb: setting up run ft7zmwyh
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261106_003314-ft7zmwyh
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run frnn_tuneB_posfrac05
wandb: ⭐️ View project at https://wandb.ai/suchok-pn-ukrainian-catholic-university/ucu-hw2-detection
wandb: 🚀 View run at https://wandb.ai/suchok-pn-ukrainian-catholic-university/ucu-hw2-detection/runs/ft7zmwyh


W&B B: https://wandb.ai/suchok-pn-ukrainian-catholic-university/ucu-hw2-detection/runs/ft7zmwyh
loaded checkpoint: fasterrcnn_baseline_last.pth
step     0 | total 1.0956 | loss_classifier=0.283 loss_box_reg=0.629 loss_objectness=0.048 loss_rpn_box_reg=0.135 | 3s
step   100 | total 0.8394 | loss_classifier=0.223 loss_box_reg=0.460 loss_objectness=0.033 loss_rpn_box_reg=0.123 | 192s
step   200 | total 1.4311 | loss_classifier=0.476 loss_box_reg=0.811 loss_objectness=0.029 loss_rpn_box_reg=0.116 | 381s
step   300 | total 1.2146 | loss_classifier=0.421 loss_box_reg=0.673 loss_objectness=0.033 loss_rpn_box_reg=0.088 | 571s
step   400 | total 1.0009 | loss_classifier=0.280 loss_box_reg=0.575 loss_objectness=0.018 loss_rpn_box_reg=0.127 | 760s
step   500 | total 1.0152 | loss_classifier=0.208 loss_box_reg=0.606 loss_objectness=0.054 loss_rpn_box_reg=0.147 | 949s
step   600 | total 0.9033 | loss_classifier=0.269 loss_box_reg=0.492 loss_objectness=0.068 loss_rpn_box_reg=0.075 | 1139s
step   700

## 5. Evaluate B on full val

Full 2,846-frame validation, same GT (`iscrowd=1` for ignore regions), same metric.
GPU-only speed benchmark. Everything logged into B's W&B run.

In [5]:
model_b.eval()
metrics_b, pred_b_json = evaluate_full_val(
    model_b, val_index, STRAT, SIZE, str(gt_fullval_ignore),
    score_thr=0.001, device=DEVICE,
    out_json=str(WORK_DIR / "val_preds_B.json"))
print("Variant B metrics:", json.dumps(metrics_b, indent=2))

speed_b = benchmark_speed_gpu(model_b, val_index, STRAT, SIZE, n=50, device=DEVICE)
print("Variant B speed:", speed_b)

run_b.log({"val/mAP": metrics_b["mAP"], "val/AP50": metrics_b["AP50"],
           "val/AP75": metrics_b["AP75"],
           "val/AP_small": metrics_b["AP_small"],
           "val/AP_medium": metrics_b["AP_medium"],
           "val/AP_large": metrics_b["AP_large"],
           "speed/gpu_fps": speed_b["gpu_fps"]})
run_b.summary["val/mAP"]       = metrics_b["mAP"]
run_b.summary["val/AP_small"]  = metrics_b["AP_small"]
run_b.summary["speed/gpu_fps"] = speed_b["gpu_fps"]
run_b.finish()
print("Experiment B: W&B run finished.")

del model_b
torch.cuda.empty_cache()

pred letterbox-1280:   0%|          | 0/2846 [00:00<?, ?it/s]

Variant B metrics: {
  "mAP": 0.15347873584867527,
  "AP50": 0.3197073201658654,
  "AP75": 0.13075500088694844,
  "AP_small": 0.0951707720531747,
  "AP_medium": 0.1642744748795781,
  "AP_large": 0.2653879767494026
}


wandb: updating run metadata


Variant B speed: {'device': 'cuda', 'gpu_ms_per_image': 349.8079013824463, 'gpu_fps': 2.858711870280758, 'std_ms': 5.033588220933071, 'n': 50}


wandb: uploading output.log; uploading config.yaml
wandb: 
wandb: Run history:
wandb:          speed/gpu_fps ▁
wandb:     train/loss_box_reg ▆▇▅▆▄█▁▆▄▂▄▇▃▆▇▅▅▃▄▆█▂▅▄▄█▃▆▂▃▅▆▆▆▄▆▃▇▇▄
wandb:  train/loss_classifier ▇▃▄▃▅▃▄▄▅▄█▂▃▃▇▇▁▅▄▁▂▃▄▅▂▃▄▅▄▇▅▂▅▃▃▇▄▅▃▄
wandb:  train/loss_objectness ▄▃▃▅▆▄▄▂▂▄▃▁▄▃▁▃▄▄▃▃▄▆▂▃▃▁▆▃█▂▄▃▃▆▁▇▄▄▇▁
wandb: train/loss_rpn_box_reg ▅▅▅▆▃█▄▅▅▁▅▆▄▄▃▃▂▆▃▃▁▄▆▃▄▅▄▅▅█▄▄▃▅▃█▆▆▇▃
wandb:             train/step ▁▁▁▁▂▂▃▃▃▃▄▄▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
wandb:            train/total █▅▂▅▅▆▃▅▅▃▄█▅▆▆▄▃▅▅▇▆▄▄▁▆▅▆▅▃▅▅▇▅▄▅▅▃▇▃▄
wandb:               val/AP50 ▁
wandb:               val/AP75 ▁
wandb:           val/AP_large ▁
wandb:                     +3 ...
wandb: 
wandb: Run summary:
wandb:          speed/gpu_fps 2.85871
wandb:     train/loss_box_reg 0.44798
wandb:  train/loss_classifier 0.29673
wandb:  train/loss_objectness 0.03452
wandb: train/loss_rpn_box_reg 0.10601
wandb:             train/step 12100
wandb:            train/total 0.88524
wandb:               val/AP50 0.31971
wan

Experiment B: W&B run finished.


## 6. Comparison table

Baseline = the sanity-check value from nb04 (`mAP=0.1475`).
Winner selected by mAP, ties broken by AP_small.

This table **is** the graded artifact for the Two-stage Anchor-based checkpoint.

In [6]:
BASELINE = {
    "variant": "baseline",
    "config":  "rpn=1000, pos_frac=0.25",
    "mAP":      0.1475, "AP50": 0.3094, "AP75": 0.1267,
    "AP_small": 0.0803, "AP_medium": 0.1570, "AP_large": 0.2666,
    "gpu_fps":  4.67,
}

rows = [
    BASELINE,
    {"variant": "A (rpn=2000)", "config": "rpn=2000, pos_frac=0.25",
     **metrics_a, "gpu_fps": speed_a["gpu_fps"]},
    {"variant": "B (pos=0.5)",  "config": "rpn=1000, pos_frac=0.50",
     **metrics_b, "gpu_fps": speed_b["gpu_fps"]},
]
df_cmp = pd.DataFrame(rows)[["variant", "config", "mAP", "AP50", "AP75",
                              "AP_small", "AP_medium", "AP_large", "gpu_fps"]].round(4)
display(df_cmp)
df_cmp.to_csv(WORK_DIR / "twostage_tuning_comparison.csv", index=False)

df_tuned = df_cmp.iloc[1:].sort_values(["mAP", "AP_small"], ascending=False)
WINNER_ROW  = df_tuned.iloc[0]
WINNER_NAME = WINNER_ROW["variant"]
print("Winner:", WINNER_NAME,
      "| mAP:", WINNER_ROW["mAP"], "| AP_small:", WINNER_ROW["AP_small"])

,variant,config,mAP,AP50,AP75,AP_small,AP_medium,AP_large,gpu_fps
0,baseline,"rpn=1000, pos_frac=0.25",0.1475,0.3094,0.1267,0.0803,0.1570,0.2666,4.6700
1,A (rpn=2000),"rpn=2000, pos_frac=0.25",0.1561,0.3234,0.1335,0.0963,0.1656,0.2706,2.8408
2,B (pos=0.5),"rpn=1000, pos_frac=0.50",0.1535,0.3197,0.1308,0.0952,0.1643,0.2654,2.8587


Winner: A (rpn=2000) | mAP: 0.1561 | AP_small: 0.0963
